In [5]:

import torch
import os


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)


DATA_DIR = "/kaggle/input/datasets/uzair2000/grapse-last/ADD_Final_Grape"
SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

TRAIN_DIR = os.path.join(DATA_DIR, "Train")   # must match exact folder name
VAL_DIR   = os.path.join(DATA_DIR, "Valid")

print("Train folder exists:", os.path.exists(TRAIN_DIR))
print("Valid folder exists:", os.path.exists(VAL_DIR))


Using Device: cuda
Train folder exists: True
Valid folder exists: True


In [9]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Hyperparameters
IMG_SIZE = 240
BATCH_SIZE = 16
EPOCHS = 80
LR = 1e-5
PATIENCE = 12

# Transforms
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Dataset paths (update according to your Kaggle dataset)
TRAIN_DIR = "/kaggle/input/datasets/uzair2000/grapse-last/ADD_Final_Grape/Train"
VAL_DIR   = "/kaggle/input/datasets/uzair2000/grapse-last/ADD_Final_Grape/Valid"
SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

# Load datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_dataset   = datasets.ImageFolder(VAL_DIR, transform=val_tfms)

# Save class names
class_names = train_dataset.classes
with open(f"{SAVE_DIR}/class_names.txt","w") as f:
    for c in class_names:
        f.write(c+"\n")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Classes:", class_names)
print("Training Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))

Classes: ['Grape Anthracnose leaf', 'Grape Black Rot', 'Grape Brown spot leaf', 'Grape Downy mildew leaf', 'Grape Mites_leaf disease', 'Grape Normal_leaf', 'Grape Powdery_mildew leaf', 'Grape shot hole leaf disease']
Training Images: 7080
Validation Images: 1900


In [10]:
!pip install timm -q

In [12]:
import os
import torch
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# =========================================================
# 8. LOAD PRETRAINED EFFICIENTNET-B4 (FULL FINETUNE)
# =========================================================
num_classes = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model('efficientnet_lite0', pretrained=True, num_classes=num_classes)
model = model.to(device)

# FULL FINETUNING (no freezing)
for param in model.parameters():
    param.requires_grad = True

# =========================================================
# 9. LOSS & OPTIMIZER
# =========================================================
LR = 1e-4
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    patience=3,
    factor=0.3
)

# =========================================================
# 10. TRAIN FUNCTION
# =========================================================
def train_one_epoch():
    model.train()
    total_loss, correct, total = 0, 0, 0

    loop = tqdm(train_loader, desc="Training", leave=False)

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item())

    return total_loss / len(train_loader), 100 * correct / total

# =========================================================
# 11. VALIDATION FUNCTION
# =========================================================
def validate():
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        loop = tqdm(val_loader, desc="Validation", leave=False)
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(val_loader), 100 * correct / total, all_preds, all_labels

# =========================================================
# 12. TRAINING LOOP (SAFE AUTO-SAVE VERSION)
# =========================================================
EPOCHS = 100
PATIENCE = 15
SAVE_DIR = "./results"
os.makedirs(SAVE_DIR, exist_ok=True)
class_names = ["Grape_Anthracnose_leaf_Disease","Grape_Black_Rot_", "Grape_Brown_spot_leaf_Disease", "Grape_Downy_mildew_Disease", "Grape_Mites_Leaf_Disease", "Grape_Healthy_Leaf","Grape_Powdery_Milde_Leaf_Disease", "Grape_shot_hole_Leaf_Disease"]   

best_acc = 0
patience_counter = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
           "val_precision": [], "val_recall": []}

try:
    for epoch in range(EPOCHS):
        print(f"\nEpoch [{epoch+1}/{EPOCHS}]")

        train_loss, train_acc = train_one_epoch()
        val_loss, val_acc, preds, labels = validate()

        # Compute precision and recall for this epoch
        precision = precision_score(labels, preds, average='weighted')
        recall = recall_score(labels, preds, average='weighted')

        scheduler.step(val_acc)

        # Save metrics in history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_precision"].append(precision)
        history["val_recall"].append(recall)

        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
        print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")

        # SAVE BEST MODEL AUTOMATICALLY
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{SAVE_DIR}/best.pt")
            patience_counter = 0
            print("🔥 Best model saved")
        else:
            patience_counter += 1

        # SAVE GRAPHS AFTER EVERY EPOCH
        plt.figure(figsize=(15,5))

        # Loss & Accuracy
        plt.subplot(1,3,1)
        plt.plot(history["train_loss"], label="Train Loss")
        plt.plot(history["val_loss"], label="Val Loss")
        plt.legend()
        plt.title("Loss Curve")

        plt.subplot(1,3,2)
        plt.plot(history["train_acc"], label="Train Acc")
        plt.plot(history["val_acc"], label="Val Acc")
        plt.legend()
        plt.title("Accuracy Curve")

        # Precision & Recall
        plt.subplot(1,3,3)
        plt.plot(history["val_precision"], label="Precision")
        plt.plot(history["val_recall"], label="Recall")
        plt.legend()
        plt.title("Precision & Recall Curve")

        plt.tight_layout()
        plt.savefig(f"{SAVE_DIR}/metrics_graph.png")
        plt.close()

        # EARLY STOPPING
        if patience_counter >= PATIENCE:
            print("⛔ Early stopping triggered")
            break

except KeyboardInterrupt:
    print("\n🛑 Training stopped manually by user!")

finally:
    print("\n💾 Saving final model and evaluation files...")

    # Save last model no matter what
    torch.save(model.state_dict(), f"{SAVE_DIR}/last.pt")

    # Load best model safely
    if os.path.exists(f"{SAVE_DIR}/best.pt"):
        model.load_state_dict(torch.load(f"{SAVE_DIR}/best.pt", map_location=device))
        print("✅ Best model loaded for final evaluation")

    # Final Validation
    val_loss, val_acc, preds, labels = validate()

    # =========================================================
    # SAVE FINAL METRICS
    # =========================================================
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='weighted')
    recall = recall_score(labels, preds, average='weighted')
    f1 = f1_score(labels, preds, average='weighted')

    metrics_df = pd.DataFrame({
        "Accuracy": [accuracy],
        "Precision": [precision],
        "Recall": [recall],
        "F1-Score": [f1],
        "Best_Val_Accuracy": [best_acc]
    })
    metrics_df.to_csv(f"{SAVE_DIR}/final_metrics.csv", index=False)

    # Classification Report
    report = classification_report(labels, preds, target_names=class_names)
    with open(f"{SAVE_DIR}/classification_report.txt", "w") as f:
        f.write(report)

    # =========================================================
    # CONFUSION MATRIX
    # =========================================================
    cm = confusion_matrix(labels, preds)
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    plt.figure(figsize=(18,14))
    sns.heatmap(cm, annot=True, fmt='.2f',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.title("Confusion Matrix")
    plt.savefig(f"{SAVE_DIR}/confusion_matrix.png")
    plt.close()

    print("✅ ALL FILES SAVED IN", SAVE_DIR)

model.safetensors:   0%|          | 0.00/18.8M [00:00<?, ?B/s]


Epoch [1/100]


Train Loss: 0.6034 | Val Loss: 0.3133
Train Acc: 81.03% | Val Acc: 90.74%
Precision: 0.9109 | Recall: 0.9074
🔥 Best model saved

Epoch [2/100]


Train Loss: 0.1924 | Val Loss: 0.2270
Train Acc: 93.55% | Val Acc: 93.26%
Precision: 0.9374 | Recall: 0.9326
🔥 Best model saved

Epoch [3/100]


Train Loss: 0.1390 | Val Loss: 0.2564
Train Acc: 95.14% | Val Acc: 93.05%
Precision: 0.9348 | Recall: 0.9305

Epoch [4/100]


...

Train Loss: 0.0090 | Val Loss: 0.2577
Train Acc: 99.70% | Val Acc: 95.00%
Precision: 0.9511 | Recall: 0.9500

Epoch [34/100]


Train Loss: 0.0095 | Val Loss: 0.2714
Train Acc: 99.66% | Val Acc: 94.89%
Precision: 0.9514 | Recall: 0.9489

Epoch [35/100]


Train Loss: 0.0082 | Val Loss: 0.2819
Train Acc: 99.73% | Val Acc: 94.95%
Precision: 0.9523 | Recall: 0.9495

Epoch [36/100]


Train Loss: 0.0088 | Val Loss: 0.2834
Train Acc: 99.68% | Val Acc: 94.84%
Precision: 0.9525 | Recall: 0.9484
⛔ Early stopping triggered

💾 Saving final model and evaluation files...
✅ Best model loaded for final evaluation


✅ ALL FILES SAVED IN ./results
